# Data Validation

## Business Objective

Before conducting exploratory analysis, the consistency and integrity of the datasets must be validated.

This notebook verifies relationships between tables, checks business rules and identifies potential data inconsistencies that could affect analytical results.

## Notebook Objectives

- Validate referential integrity
- Verify business rules
- Check datetime consistency
- Detect invalid values
- Validate relationships between datasets
- Summarize validation results

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from IPython.display import display

In [2]:
DATA_PATH = Path("../06_Datasets/raw")

dataset_files = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "categories": "product_category_name_translation.csv"
}

datasets = {}
for name, filename in dataset_files.items():
    datasets[name] = pd.read_csv(DATA_PATH / filename)

In [3]:
datasets["orders"].dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [4]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

datasets["orders"][date_columns] = (
    datasets["orders"][date_columns]
    .apply(pd.to_datetime)
)
datasets["orders"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


### Datetime Conversion

Order-related timestamps were converted from object to datetime format.

This enables chronological validation and time-based analysis in subsequent notebooks.

In [5]:
invalid_customers = (
    ~datasets["orders"]["customer_id"].isin(
        datasets["customers"]["customer_id"]
    )
).sum()

print(f"Orders with invalid customers: {invalid_customers}")

Orders with invalid customers: 0


**Result**

All orders reference valid customers.

No orphan customer records were identified.

In [17]:
invalid_products = (
    ~datasets["order_items"]["product_id"].isin(
        datasets["products"]["product_id"]
    )
).sum()

print(f"Products without product item: {invalid_products}")

Products without product item: 0


In [19]:
invalid_sellers = (
    ~datasets["order_items"]["seller_id"].isin(
        datasets["sellers"]["seller_id"]
    )
).sum()

print(f"Order item with invalid seller id: {invalid_sellers}")

Order item with invalid seller id: 0


In [20]:
invalid_reviews = (
    ~datasets["order_reviews"]["order_id"].isin(
        datasets["orders"]["order_id"]
    )
).sum()

print(f"Reviews with invalid order id: {invalid_reviews}")

Reviews with invalid order id: 0


In [21]:
invalid_payments = (
    ~datasets["order_payments"]["order_id"].isin(
        datasets["orders"]["order_id"]
    )
).sum()

print(f"Payments with invalid order id: {invalid_reviews}")

Payments with invalid order id: 0


In [16]:
validation_results = pd.DataFrame({
    "Relationship": [
        "Orders → Customers",
        "Order Items → Products",
        "Order Items → Sellers",
        "Order Reviews → Orders",
        "Order Payments → Orders"
    ],
    "Invalid References": [
        invalid_customers,
        invalid_products,
        invalid_sellers,
        invalid_reviews,
        invalid_payments
    ]
})

validation_results["Status"] = validation_results["Invalid References"].apply(
    lambda x: "✅ Passed" if x == 0 else "⚠ Review"
)

display(validation_results)

,Relationship,Invalid References,Status
0,Orders → Customers,0,✅ Passed
1,Order Items → Products,0,✅ Passed
2,Order Items → Sellers,0,✅ Passed
3,Order Reviews → Orders,0,✅ Passed
4,Order Payments → Orders,0,✅ Passed


## Referential Integrity Summary

All foreign key relationships were successfully validated.

No orphan records were detected across the analyzed datasets.

The datasets demonstrate strong referential integrity and are suitable for further business analysis.

## Datetime Consistency Validation

This section verifies whether the order lifecycle follows a valid chronological sequence.

The objective is to identify impossible or inconsistent timestamps that may indicate data quality issues.

In [23]:
invalid_approval = datasets["orders"][
    datasets["orders"]["order_approved_at"]
    < datasets["orders"]["order_purchase_timestamp"]
]

print("Unexpected approvals before purchase:", len(invalid_approval))

Unexpected approvals before purchase: 0


In [24]:
invalid_carrier = datasets["orders"][
    datasets["orders"]["order_delivered_carrier_date"]
    < datasets["orders"]["order_approved_at"]
]

print("Unexpected deliveries to carrier before approving:", len(invalid_carrier))

Unexpected deliveries before approving: 1359


In [25]:
invalid_delivery = datasets["orders"][
    datasets["orders"]["order_delivered_customer_date"]
    < datasets["orders"]["order_delivered_carrier_date"]
]

print("Unexpected deliveries to customer before delivery to carrier:", len(invalid_delivery))

Unexpected deliveries to customer before delivery to carrier: 23


In [26]:
invalid_estimated = datasets["orders"][
    datasets["orders"]["order_estimated_delivery_date"]
    < datasets["orders"]["order_purchase_timestamp"]
]

print("Unexpected estimated delivery date before date of purchase:", len(invalid_estimated))

Unexpected estimated delivery date before date of purchase: 0


In [27]:
datetime_validation = pd.DataFrame({
    "Validation": [
        "Approval before Purchase",
        "Carrier before Approval",
        "Customer before Carrier",
        "Estimated Delivery before Purchase"
    ],
    "Invalid Records": [
        len(invalid_approval),
        len(invalid_carrier),
        len(invalid_delivery),
        len(invalid_estimated)
    ]
})

datetime_validation["Status"] = datetime_validation["Invalid Records"].apply(
    lambda x: "✅ Passed" if x == 0 else "⚠ Review"
)

display(datetime_validation)

,Validation,Invalid Records,Status
0,Approval before Purchase,0,✅ Passed
1,Carrier before Approval,1359,⚠ Review
2,Customer before Carrier,23,⚠ Review
3,Estimated Delivery before Purchase,0,✅ Passed


### Datetime Validation

**Observation**

Two chronological inconsistencies were identified:

- 1,359 orders where the carrier handoff timestamp precedes the approval timestamp.
- 23 orders where customer delivery precedes the carrier handoff timestamp.

**Possible Explanations**

- asynchronous event logging;
- timestamp synchronization delays;
- data quality issues.

**Next Step**

Investigate the time differences before deciding whether these records should be corrected or excluded.

In [31]:
invalid_carrier[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].head(10)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32
64,688052146432ef8253587b930b01a06d,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39
481,32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-04 16:49:21,2018-07-05 16:33:06,2018-07-05 14:50:00,2018-07-07 14:41:18
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-24 11:32:11,2018-07-29 23:30:52,2018-07-26 14:46:00,2018-07-27 18:55:57
585,16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 01:09:09,2018-05-07 16:52:39,2018-05-07 15:09:00,2018-05-24 00:31:18
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37
817,6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-03 23:40:16,2018-07-05 16:31:26,2018-07-04 12:14:00,2018-07-05 22:52:28


In [32]:
invalid_carrier["approval_delay"] = (
    invalid_carrier["order_approved_at"]
    - invalid_carrier["order_delivered_carrier_date"]
)

invalid_carrier["approval_delay"].describe()

D:\TEMP\Temp2\ipykernel_16536\419079243.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  invalid_carrier["approval_delay"] = (


count                         1359
mean     1 days 00:45:07.153053715
std      4 days 19:18:28.055754668
min                0 days 00:00:21
25%         0 days 01:24:55.500000
50%                0 days 17:10:04
75%                1 days 01:57:24
max              171 days 05:15:22
Name: approval_delay, dtype: object

In [33]:
invalid_delivery[
    [
        "order_id",
        "order_status",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
]

invalid_delivery["delivery_diff"] = (
    invalid_delivery["order_delivered_customer_date"]
    - invalid_delivery["order_delivered_carrier_date"]
)

invalid_delivery["delivery_diff"].describe()

D:\TEMP\Temp2\ipykernel_16536\1522239141.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  invalid_delivery["delivery_diff"] = (


count                             23
mean     -4 days +17:32:41.521739131
std        3 days 17:18:29.050924326
min               -17 days +21:41:31
25%         -6 days +15:14:27.500000
50%                -2 days +08:08:08
75%                -1 days +00:39:42
max                -1 days +23:36:42
Name: delivery_diff, dtype: object

### Datetime Validation Results

**Observation**

Two types of chronological inconsistencies were identified:

- **1,359** orders where the carrier handoff timestamp precedes the approval timestamp.
- **23** orders where the customer delivery timestamp precedes the carrier handoff timestamp.

**Analysis**

The first anomaly ranges from **21 seconds** to **171 days**, with a median difference of approximately **17 hours**. This indicates that the inconsistencies cannot be explained solely by minor system synchronization delays.

The second anomaly affects only **23** orders but violates the expected order lifecycle and should be treated as inconsistent data.

**Decision**

These records will be retained in the dataset.

The anomalies will be documented and considered during subsequent analyses, but their low frequency suggests they are unlikely to materially affect aggregate business insights.

In [34]:
invalid_carrier["order_status"].value_counts()

order_status
delivered    1350
shipped         9
Name: count, dtype: int64

In [35]:
invalid_delivery["order_status"].value_counts()

order_status
delivered    23
Name: count, dtype: int64

# Data Quality Assessment Summary

## Executive Summary

A comprehensive data validation was performed before exploratory analysis.

The objective was to assess dataset quality, verify relationships between tables and identify issues that could influence business conclusions.

The validation included:

- Missing value analysis
- Duplicate detection
- Referential integrity checks
- Datetime consistency validation
- Business rule verification

In [37]:
data_quality_report = pd.DataFrame({
    "Validation Area": [
        "Missing Values",
        "Duplicate Records",
        "Referential Integrity",
        "Datetime Consistency"
    ],

    "Findings": [
        "Missing values identified in Orders, Reviews and Products",
        "261,831 fully duplicated rows in Geolocation",
        "No orphan records detected",
        "Chronological anomalies detected"
    ],

    "Decision": [
        "Keep and document",
        "Remove full duplicates only",
        "Passed",
        "Keep and document"
    ],

    "Status": [
        "✅",
        "✅",
        "✅",
        "⚠"
    ]

})

display(data_quality_report)

,Validation Area,Findings,Decision,Status
0,Missing Values,"Missing values identified in Orders, Reviews a...",Keep and document,✅
1,Duplicate Records,"261,831 fully duplicated rows in Geolocation",Remove full duplicates only,✅
2,Referential Integrity,No orphan records detected,Passed,✅
3,Datetime Consistency,Chronological anomalies detected,Keep and document,⚠


## Overall Assessment

The datasets demonstrate a high level of quality and consistency.

Referential integrity is fully preserved across all core tables.

Missing values were determined to be either expected or requiring further investigation rather than immediate imputation.

Duplicate records are limited to fully identical observations within the Geolocation dataset and can be safely removed.

A small number of chronological inconsistencies were identified during timestamp validation. Although these anomalies violate the expected order lifecycle, their frequency is low and they are unlikely to materially affect aggregate business analyses.

Overall, the datasets are considered suitable for exploratory analysis after minor preprocessing.

## Risks and Limitations

The following limitations should be considered during further analysis:

- Missing product metadata may limit category-level analyses.
- Datetime anomalies may slightly affect delivery time calculations.
- Geolocation duplicates should be removed before spatial analyses.
- Results involving small subsets of anomalous orders should be interpreted with caution.

# Next Step

The next notebook focuses on Exploratory Data Analysis (EDA).

The objective will be to transform validated data into business insights by exploring customer behavior, sales dynamics, seller performance and product characteristics.